# 03. GARCH done right

Milestone 3: GARCH(1,1), GJR-GARCH and EGARCH fitted on **S&P 500 daily log returns**, forecasting
the realised volatility of the next 21 sessions (`rv21`, ADR 0001 4.2), and compared with VIX as a
forecast of the same quantity; ARIMA-GARCH (an AR mean on ln VIX, GARCH on its innovations) as a
forecaster of the VIX level; and why GARCH is never fitted on the VIX level itself.

The 21-session forecast is the sum of the h-step conditional variances, annualised like `rv21`:

```
G_t = (252/21) * sum_{k=1}^{21} sigma2_{t+k|t} / 100**2      yhat = 100 * sqrt(G_t)
```

(`arch` is fitted on returns x 100, so its variances are in percent squared.) The full run:

```
uv run python -m ivf.models garch            # real data, local only
uv run python -m ivf.models --synthetic garch
```

This notebook runs the last `IVF_NOTEBOOK_FOLDS` dev folds (default 4). **Licensing:** real-data
numbers stay on your machine (ADR 0001 D2).

*This file is generated by `tools/build_notebooks.py`. Edit the spec there.*

In [ ]:
import json
import math
import os
import warnings

import numpy as np
import pandas as pd

from ivf import config, protocol
from ivf.data import build_interim, calendar, check, synthetic
from ivf.evaluation import walkforward
from ivf.features import build
from ivf.models import baselines, garch, registry, rv21

warnings.filterwarnings("ignore", category=FutureWarning)


def load():
    """(mode, protocol, iv, price, build_report): real if available, else synthetic."""
    if os.environ.get("IVF_NOTEBOOK_DATA") != "synthetic":
        try:
            proto = protocol.load_protocol(protocol.protocol_path())
            proto.require_data_end()
            iv, price, report = build_interim.read_interim(config.data_layout().root)
            return "real", proto, iv, price, report
        except (protocol.ProtocolError, build_interim.InterimError) as why:
            print("no real data:", why)
    proto = protocol.load_protocol(protocol.protocol_path(synthetic=True))
    iv, price = synthetic.synthetic_market(proto)
    report = {"files": [{"symbol": s, "fetched_at": "2026-10-27T12:00:00.000000Z", "duplicates_dropped": 0, "duplicate_dates": []}
                        for s in list(iv["symbol"].unique()) + ["^GSPC"]]}
    return "synthetic", proto, iv, price, report


MODE, PROTO, IV, PRICE, REPORT = load()
if not check.frozen_check(PROTO, IV, PRICE, REPORT).ok:
    raise RuntimeError("check reports VIOLATED: run `uv run python -m ivf.data check` and fix the data first")
CAL = calendar.master_calendar(PROTO)
CALENDAR_DATES = list(CAL["date"])
DEV = [f for f in walkforward.fold_schedule(CALENDAR_DATES, PROTO.dev_start, PROTO.holdout_start, PROTO.fold_len, PROTO.data_end) if f.period == "dev"]
DATES, _, IV_PRE, PRICE_PRE = walkforward.pre_holdout_inputs(CAL, IV, PRICE, PROTO.holdout_start)
VIX = build.align(DATES, IV_PRE, "symbol", "VIX")
INPUTS = rv21.Rv21Inputs(build.log_returns(build.align(DATES, PRICE_PRE, "ticker", "^GSPC")), VIX)
N_FOLDS = int(os.environ.get("IVF_NOTEBOOK_FOLDS", "4"))
print("data:", MODE.upper(), "| holdout_start", PROTO.holdout_start, "| sessions readable", len(DATES), "| last", DATES[-1])

## 1. The aggregation, by hand

GARCH(1,1) with omega 0.02, alpha 0.08, beta 0.90 has a long-run daily variance of
0.02 / (1 - 0.98) = 1 percent squared. From a one-step variance of 2, the path decays towards 1 at
rate 0.98 a session; 21 steps sum to 21 + (1 - 0.98^21)/0.02 = 38.287.

In [ ]:
path = garch.variance_path("garch", {"omega": 0.02, "alpha": 0.08, "beta": 0.90}, np.array([2.0]))
G = garch.aggregate(path)[0]
print(f"sum of the 21 daily variances {path.sum():.6f} (percent^2); G = 12 x sum / 10^4 = {G:.6f}; yhat = {100 * math.sqrt(G):.4f} vol points")
print("units check: 21 returns of 1% give rv21 =", round(float(build.rv_trailing(np.full(21, 0.01))[-1]), 6),
      "and a constant 1 percent^2 gives", round(100 * math.sqrt(garch.aggregate(np.ones((1, 21)))[0]), 6))

## 2. The rv21 track

Every model is fitted at each cutoff on data dated up to the cutoff. For `har_rv`, the only rv21
model with (origin, target) training pairs, an origin counts only if its 21-session target window
ends by the cutoff (rule L2 for a window target). Each GARCH-family fit tries normal and Student-t
innovations and keeps the lower AIC.

In [ ]:
RUN = walkforward.run_rv21(INPUTS, DATES, CALENDAR_DATES, DEV[-N_FOLDS:], rv21.build_rv21_models(), PROTO.holdout_start, 504)
FC = RUN.forecasts
for _, row in RUN.fits[RUN.fits.model_id.isin(garch.KINDS)].iterrows():
    p = json.loads(row.params_json)
    print(f"{row.cutoff_date} {row.model_id:6} dist {p['dist']:6} persistence {p['persistence']:.4f} AIC {row.aic:.1f}")
METRIC = walkforward.metric_table(FC, PROTO.holdout_start, PROTO.document["min_cell_n"], set())
print(METRIC.set_index("model_id")[["n", "rmse", "qlike", "bias"]].round(4))

## 3. Synthetic only: the true expected variance as a floor

The synthetic returns are a GARCH(1,1) with known parameters, so the true `E_t[v]` of the next 21
sessions is known: run the true recursion through the returns, then aggregate. Its QLIKE is the floor.

In [ ]:
if MODE == "synthetic":
    sp = synthetic.SyntheticParams()
    r = np.nan_to_num(INPUTS.r)
    h = np.empty(len(r) + 1)
    h[0] = sp.daily_variance
    for i in range(len(r)):
        h[i + 1] = sp.omega + sp.alpha * r[i] ** 2 + sp.beta * h[i]  # h[i + 1] = Var(r_{i+1} | r up to i)
    true = {"omega": sp.omega * 1e4, "alpha": sp.alpha, "beta": sp.beta}
    rows = walkforward.scored(FC[FC.model_id == "garch"], PROTO.holdout_start)
    idx = np.array([DATES.index(d) for d in rows["origin_date"]])
    g_true = garch.aggregate(garch.variance_path("garch", true, h[idx + 1] * 1e4))
    from ivf.evaluation import losses
    print(f"truth: QLIKE {losses.qlike(rows['y'].to_numpy(), 100 * np.sqrt(g_true)):.4f}, "
          f"RMSE {losses.rmse(rows['y'].to_numpy(), 100 * np.sqrt(g_true)):.4f} (n = {len(rows)})")
else:
    print("real data: no known truth")

## 4. ARIMA-GARCH on ln VIX

An AR(p) with a constant on 100 x ln VIX (d = 0, so it reverts to its mean), GARCH(1,1) on its
innovations; p and the distribution by AIC. The forecast of ln VIX at h iterates the AR; its variance
adds up the GARCH path weighted by the AR's impulse response, and the lognormal mean is scored (the
median is stored too). Compare with the random walk.

In [ ]:
VIX_RUN = walkforward.run_symbol("VIX", "vix_level", VIX, DATES, CALENDAR_DATES, DEV[-N_FOLDS:],
                                 [baselines.RandomWalk(), garch.ArimaGarch()], PROTO.horizons, PROTO.holdout_start, 504)
for _, row in VIX_RUN.fits[VIX_RUN.fits.model_id == "arima_garch_log"].iterrows():
    p = json.loads(row.params_json)
    print(f"cutoff {row.cutoff_date}: AR({p['p']}) with a constant on 100 x ln VIX, dist {p['dist']}, GARCH persistence {p['persistence']:.4f}")
VM = walkforward.metric_table(VIX_RUN.forecasts, PROTO.holdout_start, PROTO.document["min_cell_n"], {"arima_garch_log"})
print(VM.pivot_table(index=["horizon", "model_id", "forecast"], values=["rmse", "qlike"]).round(4))

## 5. Why GARCH is never fitted on the VIX level

GARCH models the variance of a zero-mean *shock*. The VIX level is not a shock: it is already a
volatility forecast, positive and persistent. Fit GARCH to it anyway, and the level's persistence
reappears as "volatility clustering", the standardised residuals stay autocorrelated, and the mean
forecast is the historical mean. On synthetic data the truth is known: ln VIX has **constant**
innovation variance, so there is no clustering to find (concept note 10). Section 4's model is the
right way round: the AR takes the level's persistence, GARCH sees only the innovations.

In [ ]:
first = int(np.flatnonzero(np.isnan(VIX))[-1]) + 1 if np.isnan(VIX).any() else 0
DEMO = garch.garch_on_the_level_demo(VIX[first:])
for key, value in DEMO.items():
    print(f"{key:28} {value:.4f}")